## Object Tracking

Given an object that we have already found in one frame (e.g. by a detector), **tracking** localises *the same object* in the following frames. Tracking is cheaper than detection because we know where the object was and what it looked like: we only have to search in the neighbourhood of the previous position.

Every tracker is built from the same two ingredients:

* a **motion model**: where can the object be in the next frame?
* an **appearance model**: what does the object look like, so we can recognise it?

In this notebook we walk through the idea step by step:

1. **Block matching**: the simplest appearance model. We write it ourselves.
2. **Why block matching is not enough**: the object changes its size.
3. **OpenCV trackers** (MIL, KCF, CSRT) and a modern **deep-learning tracker (ViT)**, compared on the same sequence.

### Data

We use frames from the [Udacity self-driving car dataset (CrowdAI)](https://www.kaggle.com/soumikrakshit/udacity-car-dataset-crowdai), i.e. a sequence of images taken by a camera mounted on a car. Download the dataset and extract the images into `data/object-detection-crowdai/` (relative to this notebook). The file names are timestamps, so sorting them alphabetically gives the frames in temporal order.

In OpenCV a bounding box is written as `(x, y, w, h)`: the top-left corner followed by the width and the height. We keep this convention throughout the notebook.

In [ ]:
import os
import time
import cv2
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

### Helper functions

OpenCV works with BGR images, so we keep the frames in BGR for all the algorithms and convert to RGB only when we display them.

In [ ]:
folder = '/run/media/janko/DATA/Datasets/object-detection-crowdai/'
frames = sorted(os.listdir(folder))


def load(idx):
    return cv2.imread(os.path.join(folder, frames[idx]))


def draw_box(img, box, color=(0, 255, 0), thickness=2):
    x, y, w, h = [int(v) for v in box]
    return cv2.rectangle(img.copy(), (x, y), (x + w, y + h), color, thickness)


def crop_around(img, box, half=120):
    # Cut a (2 * half) x (2 * half) window centred on the box, so the small object is actually visible
    # (a tracker that has lost the object can wander off, hence max() keeps the window inside the image)
    x, y, w, h = box
    cx, cy = x + w // 2, y + h // 2
    return img[max(cy - half, 0):cy + half, max(cx - half, 0):cx + half]


def show_grid(images, titles, cols):
    rows = int(np.ceil(len(images) / cols))
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()

### The sequence

We will use two situations:

* **A**: a car close to us, tracked for 10 frames.
* **B**: a small, distant car, tracked for 50 frames. As the car and our camera move, the size of the car in the image changes (we will see why this matters in a moment).

Let's pretend that a detector has already found the car in the first frame of each situation. These are the starting bounding boxes.

In [ ]:
start_a = frames.index('1479498704988166448.jpg')
box_a = (910, 545, 115, 110)

start_b = frames.index('1479498995506866553.jpg')
box_b = (965, 555, 50, 40)

plt.subplot(121), plt.imshow(cv2.cvtColor(draw_box(load(start_a), box_a, thickness=4), cv2.COLOR_BGR2RGB)), plt.title('Situation A')
plt.subplot(122), plt.imshow(cv2.cvtColor(draw_box(load(start_b), box_b, thickness=4), cv2.COLOR_BGR2RGB)), plt.title('Situation B')

## 1. Block matching

**Block matching** is the simplest appearance model: we cut the object out of the first frame and call it the **template**. In each new frame we slide the template over a **search window** around the previous position and, for every candidate position, measure how different the candidate is from the template. The position with the **smallest loss** is the new position of the object.

* The search window encodes our **motion model**: a car cannot move arbitrarily far between two consecutive frames, so we only look `search` pixels around the previous position. This makes tracking much cheaper than searching the whole frame.
* The loss is the **SAD** (sum of absolute differences) between the template and the candidate. It is cheap, because it needs no multiplications. Another popular choice is the SSD (sum of squared differences).

Let's cut out the template for situation A.

In [ ]:
x, y, w, h = box_a
template = load(start_a)[y:y+h, x:x+w].astype(np.float32) / 255

plt.imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))

The search itself is two nested loops: for every row and column offset inside the search window we cut out a candidate, compute its SAD loss and remember the best one.

In [ ]:
def block_matching(template, window):
    # Return the (column, row) offset in the window where the SAD loss is the smallest
    th, tw = template.shape[:2]
    best_loss, best_c, best_r = np.inf, 0, 0
    for r in range(window.shape[0] - th + 1):
        for c in range(window.shape[1] - tw + 1):
            candidate = window[r:r+th, c:c+tw]
            loss = np.sum(np.abs(template - candidate))
            if loss < best_loss:
                best_loss, best_c, best_r = loss, c, r
    return best_c, best_r

Now the tracking loop. For every frame we

1. cut out the search window around the **previous** box,
2. find the best matching position of the template in it,
3. convert the position from window coordinates back to image coordinates. This is the new box.

The size of the box never changes, since it is the size of the template.

In [ ]:
def track(matcher, start, box, n_frames, search=50):
    x, y, w, h = box
    template = load(start)[y:y+h, x:x+w].astype(np.float32) / 255
    boxes = []
    for i in range(start, start + n_frames):
        img = load(i)
        window = img[y-search:y+h+search, x-search:x+w+search].astype(np.float32) / 255
        c, r = matcher(template, window)
        x, y = x - search + c, y - search + r
        boxes.append((x, y, w, h))
    return boxes

In [ ]:
%%time
boxes_sad = track(block_matching, start_a, box_a, n_frames=10)

In [ ]:
n_show = [0, 3, 6, 9]
images = [crop_around(draw_box(load(start_a + i), boxes_sad[i]), boxes_sad[i]) for i in n_show]
show_grid(images, [f'Frame {i}' for i in n_show], cols=4)

### Making it faster

The two Python loops are slow, and a real tracker has to run on every frame. OpenCV can do the same sliding-window comparison in compiled code: `cv2.matchTemplate` returns the loss for **all** candidate positions at once and `cv2.minMaxLoc` finds the best one. OpenCV offers the squared difference (`TM_SQDIFF`, i.e. SSD) rather than SAD, so the loss is slightly different. Let's check how much the boxes differ and how long it takes.

In [ ]:
def block_matching_cv(template, window):
    scores = cv2.matchTemplate(window, template, cv2.TM_SQDIFF)
    _, _, (c, r), _ = cv2.minMaxLoc(scores)  # location of the smallest loss
    return c, r

In [ ]:
%%time
boxes_cv = track(block_matching_cv, start_a, box_a, n_frames=10)

In [ ]:
difference = np.abs(np.array(boxes_sad) - np.array(boxes_cv))
print('Largest difference between the SAD and SSD boxes [px]:', difference.max())

The boxes are not necessarily identical, because SAD and SSD can rank two candidates differently, but they are practically the same, and `matchTemplate` is much faster.

## 2. The problem with block matching: changing appearance

Our template is a snapshot of the object in the **first** frame, and it is never updated. Block matching therefore assumes that the object looks the same in every frame. This is rarely true:

* the **scale** changes when the object moves towards or away from the camera,
* the **background** inside the box changes,
* the **lighting** changes,
* the object is **rotated** or partially hidden.

Situation B shows the first problem. The box has a fixed size of 50×40 pixels, but the car's size in the image changes as the distance to our camera changes. We track it for 50 frames (we use the fast `matchTemplate` version).

In [ ]:
%%time
n_frames = 50
boxes_bm = track(block_matching_cv, start_b, box_b, n_frames)

In [ ]:
n_show = [0, 10, 20, 30, 40, 49]
images = [crop_around(draw_box(load(start_b + i), boxes_bm[i]), boxes_bm[i]) for i in n_show]
show_grid(images, [f'Frame {i}' for i in n_show], cols=3)

Look at the last frames: does the box still cover the car? What would a tracker need in order to follow the car correctly? It would have to adapt the **size** of the box, not only its position.

This is the motivation for more robust appearance models. OpenCV ships several ready-made trackers.

## 3. OpenCV trackers

All OpenCV single-object trackers share the same two-call interface:

```python
tracker = cv2.TrackerCSRT_create()
tracker.init(first_frame, bbox)         # tell the tracker what to follow
ok, bbox = tracker.update(next_frame)   # find it in the next frame
```

We compare four of them:

| Tracker | Idea |
|---|---|
| **MIL** | Multiple Instance Learning: trains an online classifier "object vs. background" |
| **KCF** | Kernelised Correlation Filter: learns a filter in the Fourier domain, very fast |
| **CSRT** | Correlation filter with channel and spatial reliability: slower than KCF, but usually more accurate. A good default |
| **ViT** | A modern **deep-learning** tracker based on a Vision Transformer. It runs through OpenCV's DNN module |

The first three are classical methods and work out of the box. The ViT tracker needs a pretrained network in ONNX format (less than 1 MB) that you have to download once, [`object_tracking_vittrack_2023sep.onnx`](https://github.com/opencv/opencv_zoo/tree/main/models/object_tracking_vittrack), and place next to this notebook. OpenCV has a few more DNN-based trackers (e.g. `TrackerNano`, `TrackerDaSiamRPN`) that are used the same way.

In [ ]:
vit_params = cv2.TrackerVit_Params()
vit_params.net = 'object_tracking_vittrack_2023sep.onnx'

trackers = {'MIL': cv2.TrackerMIL_create,
            'KCF': cv2.TrackerKCF_create,
            'CSRT': cv2.TrackerCSRT_create,
            'ViT': lambda: cv2.TrackerVit_create(vit_params)}

We run every tracker on situation B, starting from the same box, and store its boxes. We also measure how long the trackers spend in `update`, and count the frames where a tracker reports that it has lost the object (`ok == False`).

In [ ]:
results = {}
for name, create in trackers.items():
    tracker = create()
    tracker.init(load(start_b), box_b)

    boxes, oks, elapsed = [box_b], [True], 0
    for i in range(start_b + 1, start_b + n_frames):
        img = load(i)
        t0 = time.perf_counter()
        ok, box = tracker.update(img)
        elapsed += time.perf_counter() - t0
        boxes.append(box)
        oks.append(ok)

    results[name] = {'boxes': boxes, 'ok': oks, 'ms': 1000 * elapsed / (n_frames - 1)}
    print(f'{name:5s} {results[name]["ms"]:6.1f} ms/frame, lost in {oks.count(False)} frames')

Let's look at the result. The first row is block matching from the previous section, the other rows are the trackers. Every picture is cropped around the box of the respective tracker, so a tracker that follows the car keeps it in the centre of the picture.

A tracker that reports a loss (`ok == False`) returns an empty box `(0, 0, 0, 0)`. We do not draw it: for display we keep the last known position of the tracker and mark the frame as *lost*.

In [ ]:
all_boxes = {'Block matching': boxes_bm, **{name: r['boxes'] for name, r in results.items()}}
all_ok = {'Block matching': [True] * n_frames, **{name: r['ok'] for name, r in results.items()}}


def last_known(boxes, oks):
    shown, last = [], boxes[0]
    for box, ok in zip(boxes, oks):
        last = box if ok else last
        shown.append(last)
    return shown


n_show = [0, 15, 30, 49]
key_frames = {i: load(start_b + i) for i in n_show}

images, titles = [], []
for name in all_boxes:
    shown = last_known(all_boxes[name], all_ok[name])
    for i in n_show:
        img = draw_box(key_frames[i], shown[i]) if all_ok[name][i] else key_frames[i]
        images.append(crop_around(img, shown[i]))
        titles.append(f'{name}, frame {i}' + ('' if all_ok[name][i] else ' (lost)'))

plt.figure(figsize=(15, 18))
show_grid(images, titles, cols=len(n_show))

What do we see?

* **Block matching** and **MIL** follow the car at the beginning, but the car grows in the image as we approach it, and their boxes slide off it.
* **KCF** reports a loss from frame 2 on and never recovers (its crops show the last known position, without a box). It is the fastest of the four trackers, but it is not robust here.
* **CSRT** and **ViT** adapt the size of the box to the growing car and keep following it.
* By the last frame the car is no longer visible. CSRT and MIL nevertheless keep drawing a box and never report a failure (`ok` is always `True`), whereas ViT reports `ok == False` from frame 44 on. So the `ok` flag is not equally informative for all trackers. This is another reason why trackers are combined with a detector.

### Comparing the boxes numerically

We have no ground truth, so we cannot say which tracker is *right*, but we can still compare the trackers with each other.

First, the **size** of the box over time. Remember that block matching can never change it. A width and height of zero means that the tracker has reported a loss.

In [ ]:
plt.subplot(121)
for name, boxes in all_boxes.items():
    plt.plot([b[2] for b in boxes], label=name)
plt.title('Box width [px]'), plt.xlabel('Frame'), plt.legend()

plt.subplot(122)
for name, boxes in all_boxes.items():
    plt.plot([b[3] for b in boxes], label=name)
plt.title('Box height [px]'), plt.xlabel('Frame'), plt.legend()

Second, the **overlap** between boxes, measured by the *intersection over union* (IoU): 1 means identical boxes, 0 means no overlap. We use CSRT as the reference and measure how much the other trackers agree with it. Note that this tells us how similar the trackers are, not how accurate they are.

In [ ]:
def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    iw = max(0, min(ax + aw, bx + bw) - max(ax, bx))
    ih = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = iw * ih
    return inter / (aw * ah + bw * bh - inter)


reference = all_boxes['CSRT']
for name, boxes in all_boxes.items():
    if name != 'CSRT':
        plt.plot([iou(a, b) for a, b in zip(boxes, reference)], label=name)
plt.title('IoU with the CSRT boxes'), plt.xlabel('Frame'), plt.ylim(0, 1.05), plt.legend()

## Watching the trackers work

Single frames hide the temporal behaviour, so let's **play the sequence** with the boxes of all four trackers drawn on it: MIL in yellow, KCF in red, CSRT in green and ViT in cyan. A tracker that has lost the object has no box and is marked *lost* in the legend.

We first choose a fixed window of the image that contains all the reported boxes (otherwise we would hardly see the small boxes in a 1920×1200 picture), draw the boxes and write them into a video file, which we then embed in the notebook with the Jupyter video player (so you can pause it and move around in time). We use the VP8 codec (`.webm`), because OpenCV usually cannot write H.264 and browsers cannot play the default MPEG-4 codec.

In [ ]:
from IPython.display import Video

names = ['MIL', 'KCF', 'CSRT', 'ViT']
colors = {'MIL': (0, 255, 255), 'KCF': (0, 0, 255), 'CSRT': (0, 255, 0), 'ViT': (255, 255, 0)}  # BGR
margin, width, fps = 40, 640, 10


def label(img, text, position, color):
    # Black outline makes the text readable on any background
    cv2.putText(img, text, position, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4)
    cv2.putText(img, text, position, cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)


# Window containing all the boxes that the trackers actually reported
reported = [b for name in names for b, ok in zip(results[name]['boxes'], results[name]['ok']) if ok]
H, W = key_frames[0].shape[:2]
x0 = max(min(b[0] for b in reported) - margin, 0)
y0 = max(min(b[1] for b in reported) - margin, 0)
x1 = min(max(b[0] + b[2] for b in reported) + margin, W)
y1 = min(max(b[1] + b[3] for b in reported) + margin, H)

height = int((y1 - y0) * width / (x1 - x0))
writer = cv2.VideoWriter('tracking.webm', cv2.VideoWriter_fourcc(*'VP80'), fps, (width, height))

for i in range(n_frames):
    img = load(start_b + i)
    for name in names:
        if results[name]['ok'][i]:
            img = draw_box(img, results[name]['boxes'][i], colors[name], thickness=4)

    view = img[y0:y1, x0:x1]
    view = cv2.resize(view, (width, height))

    # Legend (a tracker that has lost the object is marked) and frame number
    label(view, f'Frame {i}', (10, 25), (255, 255, 255))
    for k, name in enumerate(names):
        ok = results[name]['ok'][i]
        label(view, name if ok else f'{name} (lost)', (10, 55 + 25 * k), colors[name] if ok else (160, 160, 160))
    writer.write(view)

writer.release()

In [ ]:
Video('tracking.webm', embed=True, html_attributes='controls loop')

Watch the sequence a few times and compare the trackers: when does each of them start to deviate from the car, and what does the deviation look like (a sudden jump, or a slow drift)? The video is also saved as `tracking.webm` next to the notebook.

## Summary: which tracker should I use?

Ask yourself what is hard in *your* problem:

* Are there (long) **occlusions**?
* Does the object's **scale** change a lot?
* Does the object's **appearance** change?
* Does the object's **motion** change abruptly?

Rules of thumb:

* **CSRT** is a good default.
* **KCF** if speed matters more than accuracy.
* A **deep-learning tracker** (such as ViT) when the classical ones fail, e.g. under appearance change, at the price of a model file and (usually) more compute.
* In practice, a tracker is combined with a detector that periodically **resets** it, because every tracker eventually drifts (accumulated error) or loses the object (occlusion).

All of these trackers follow **one** object that we select by hand. In the homework you will compare two of them on a video of your choice.